<a href="https://colab.research.google.com/github/thanhsonnguyen-gif/ai-text-detection/blob/main/notebooks/E4_model_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E4 · So sánh mô hình · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.4, RQ3)
LR và Naive Bayes trên Spark (cùng train/val/test), ghi cả thời gian và phần cứng → **T5.6**.
RoBERTa (GPU) nằm ở notebook riêng `E4b_roberta.ipynb`, tự thêm 1 dòng vào T5.6.

**Lỗi đã sửa (04/11):** lần chạy đầu NB cho ngưỡng 1,0 và TPR = 0 vì xác suất NB bão hòa đúng 1,0
(27% văn bản người trên val). Từ nay NB được chấm bằng **log-odds** (`evaluate.score(..., logodds=True)`);
LR vẫn dùng xác suất như E1–E3. Dòng `nb|clean` cũ trong runs.csv là bằng chứng của lỗi; dòng mới có nhãn `nb|clean|logodds`.

Chạy: **Runtime → Run all** (~10 phút).

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son&Tai"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Java 21 | PySpark 4.0.5
Drive: /content/drive/MyDrive/BigData_IE212_R11_VB2/AI-Text-Project


In [ ]:
# 1. Huấn luyện LR và NB trên toàn bộ train (Spark, CPU), giữ lại cả hai mô hình
from aitext import data, models, evaluate
train, val, test = data.load_splits(spark)
train = data.with_class_weight(train).cache()
M, T = {}, {}
for clf in ["lr", "nb"]:
    t0 = time.time(); M[clf] = models.fit(train, defended=True, clf=clf); T[clf] = time.time() - t0
    print(f"{clf}: {T[clf]:.0f}s")

lr: 442s
nb: 172s


In [ ]:
# 2. Chẩn đoán trên VAL: tỷ lệ xác suất bão hòa đúng 1,0. Nhãn 0 > 5% ⇒ ngưỡng FPR 5% = 1,0 ⇒ phải dùng log-odds
diag = []
for clf, m in M.items():
    p = evaluate.score(m, val, models.text_col(True))
    for r in p.groupBy("label").agg(F.avg((F.col("score") >= 1.0).cast("double")).alias("frac")).collect():
        diag.append({"mô hình": clf, "label": r["label"], "tỷ lệ xác suất = 1,0": r["frac"]})
d = pd.DataFrame(diag).sort_values(["mô hình", "label"]); display(d.round(4))
evaluate.save_table(d, "T5.6a_prob_saturation_val",
                    notes="Lý do chấm NB bằng log-odds: xác suất NB bão hòa đúng 1,0 ở nhiều văn bản người.")

,mô hình,label,"tỷ lệ xác suất = 1,0"
1,lr,0,0.0000
0,lr,1,0.0000
3,nb,0,0.2667
2,nb,1,0.8414


In [ ]:
# 3. Ngưỡng trên VAL, đo trên TEST → T5.6
rows = []
for clf, m in M.items():
    lo = clf == "nb"          # NB: xác suất bão hòa → chấm bằng log-odds
    thr = evaluate.thr_at_fpr(evaluate.score(m, val, models.text_col(True), logodds=lo))
    r = evaluate.metrics(evaluate.score(m, test, models.text_col(True), logodds=lo), thr,
                         boundary=0.0 if lo else 0.5)
    evaluate.log_run("E4", OWNER, f"{clf}|clean" + ("|logodds" if lo else ""), r, notes=f"train {T[clf]:.0f}s, Colab CPU")
    rows.append({"mô hình": clf, "điểm": "log-odds" if lo else "xác suất", "train (s)": T[clf], "phần cứng": "Colab CPU", **r})
t56 = pd.DataFrame(rows); display(t56.round(4))
evaluate.save_table(t56, "T5.6_model_compare",
                    notes="NB chấm bằng log-odds: ngưỡng ở thang log-odds; fpr_at_0.5 của NB = FPR tại log-odds 0 (≡ p 0,5). Xem T5.6a.")

,mô hình,điểm,train (s),phần cứng,auroc,fpr_at_0.5,tpr_at_fpr_oracle,n,threshold,tpr,fpr
0,lr,xác suất,441.7950,Colab CPU,0.9131,0.4873,0.7106,15840,0.7309,0.7343,0.0627
1,nb,log-odds,171.5603,Colab CPU,0.8771,0.3412,0.5763,15840,214.0727,0.6204,0.0696


## Ghi chú cho Chương 5
- ...